In [ ]:
!pip install ultralytics kagglehub --quiet

In [ ]:
import pandas as pd
import os
import shutil
import random
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nguyenquocbaokaggle/proccessingdata")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'proccessingdata' dataset.
Path to dataset files: /kaggle/input/proccessingdata


In [ ]:
# Thư mục cache từ kagglehub
src_path = "/root/.cache/kagglehub/datasets/nguyenquocbaokaggle/proccessingdata/versions/1"

# Thư mục làm việc trong Colab
dst_path = "/content/dataset"

# Nếu folder đích đã tồn tại thì skip
if os.path.exists(dst_path):
    print(f"{dst_path} đã tồn tại, skip di chuyển dataset.")
elif os.path.exists(src_path):
    # Tạo folder đích nếu chưa có
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    # Di chuyển dataset
    shutil.move(src_path, dst_path)
    print("Dataset đã được di chuyển tới:", dst_path)
else:
    print(f"Không tìm thấy dataset tại {src_path}, vui lòng tải lại.")

/content/dataset đã tồn tại, skip di chuyển dataset.


In [ ]:
random.seed(42)

root_img_folder = "/content/dataset/enhanced_images/content/enhanced_images"
csv_file = "/content/dataset/Train_clean.csv"  # file CSV
df = pd.read_csv(csv_file)

# Tạo folder YOLO structure
dst_base = "/content/dataset/yolo_dataset"
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(dst_base, "images", split), exist_ok=True)
    os.makedirs(os.path.join(dst_base, "labels", split), exist_ok=True)

# Chia train/val/test theo ảnh (giữ class distribution)
train_val_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['ClassId'])
train_df, val_df = train_test_split(train_val_df, test_size=0.1111, random_state=42, stratify=train_val_df['ClassId'])
# 0.1111 * 0.9 ≈ 0.1 tổng → train 0.8, val 0.1, test 0.1

def link_images(df_split, split):
    for _, row in df_split.iterrows():
        # nối với root folder đúng
        src_img = os.path.join(root_img_folder, row['Path'])
        dst_img = os.path.join(dst_base, "images", split, os.path.basename(row['Path']))
        os.makedirs(os.path.dirname(dst_img), exist_ok=True)
        if not os.path.exists(dst_img):
            try:
                os.symlink(src_img, dst_img)
            except FileExistsError:
                pass

link_images(train_df, "train")
link_images(val_df, "val")
link_images(test_df, "test")

print("Tách tập train/val/test xong!")

Tách tập train/val/test xong!


In [ ]:
# Hàm ghi YOLO txt
def write_yolo_labels(df_split, split):
    for _, row in df_split.iterrows():
        w_img, h_img = row['Width'], row['Height']
        x1, y1, x2, y2 = row['Roi.X1'], row['Roi.Y1'], row['Roi.X2'], row['Roi.Y2']
        class_id = row['ClassId']

        # Chuyển sang YOLO format
        x_center = ((x1 + x2) / 2) / w_img
        y_center = ((y1 + y2) / 2) / h_img
        w_box = (x2 - x1) / w_img
        h_box = (y2 - y1) / h_img

        label_path = os.path.join(dst_base, "labels", split, os.path.basename(row['Path']).replace(".png",".txt"))
        with open(label_path, "w") as f:
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w_box:.6f} {h_box:.6f}\n")

write_yolo_labels(train_df, "train")
write_yolo_labels(val_df, "val")
write_yolo_labels(test_df, "test")

print("Convert CSV → YOLO labels xong!")

Convert CSV → YOLO labels xong!


In [ ]:
yaml_content = f"""
train: {dst_base}/images/train
val: {dst_base}/images/val
test: {dst_base}/images/test

nc: 43  # số class của traffic signs
names: [{",".join([f'"{i}"' for i in range(43)])}]
"""

yaml_path = os.path.join(dst_base, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("data.yaml đã tạo xong tại:", yaml_path)

data.yaml đã tạo xong tại: /content/dataset/yolo_dataset/data.yaml


In [ ]:
model = YOLO("yolov8s.pt")

model.train(
    data=yaml_path,
    epochs=100,
    imgsz=320,
    batch=16,
    lr0=0.01,
    optimizer="SGD",
    pretrained=True,
    project="runs/train",
    name="traffic_signs_ft"
)

Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=traffic_signs_ft9, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100, perspective=

KeyboardInterrupt: 